# Develop, Train, Optimize and Deploy Scikit-Learn Random Forest with custom scripts

In this notebook we show how to use Amazon SageMaker to develop, train, tune and deploy a Random Forest model based using the popular ML framework [Scikit-Learn].

The example uses the *California Housing dataset*.

## Set up libraries and environment

Once all relevant dependencies are installed, we can start out by importing the dependencies and setting up basic configurations, like the current AWS Region and target Amazon S3 bucket.

In [ ]:
# Python Built-Ins:

# External Dependencies:
import boto3  # General-purpose AWS SDK for Python
import pandas as pd  # Tools for warking with data tables (dataframes)
import sagemaker  # High-level SDK for Amazon SageMaker in particular

sm_boto3 = boto3.client("sagemaker")
sess = sagemaker.Session()
region = sess.boto_session.region_name
bucket = sess.default_bucket()  # this could also be a hard-coded bucket name
prefix = "sagemaker/3-custom-scripts"

print(f"Using bucket {bucket}")

## Create a custom training script

The SageMaker Scikit-Learn [Framework Container](https://docs.aws.amazon.com/sagemaker/latest/dg/pre-built-docker-containers-scikit-learn-spark.html) provides the basic runtime, and we as users specify the actual training steps to run as a script file (or even a folder of several, perhaps including a `requirements.txt`).

The code is located at [`src/main.py`](src/main.py).

In this example, the same file will be used at training time (run as as script), and at inference time (imported as a [module](https://docs.python.org/3/tutorial/modules.html)) - So below we:

- Define some specific **inference functions** to override default behavior (e.g. `model_fn()`), and
- Enclose the **training entry point** in an `if __name__ == '__main__'` [guard clause](https://docs.python.org/3/library/__main__.html) so it only executes when the module is run as a script.

## Local training

Since configuration is by command line arguments, we can test our training script locally before uploading to a SageMaker training job.

> ⚠️ **Note:** This is good for quick, functional tests of your script against small sample datasets... But once you're confident your script *functionally* works, you probably want to move your experiments to reproduceable, trackable, SageMaker training jobs. Be aware that the libraries in your notebook kernel may not exactly match the container image you configure for the training job later.

In [ ]:
!mkdir model/

!python src/main.py \
    --n_estimators 100 \
    --min_samples_leaf 3 \
    --model_dir model/ \
    --train data/train \
    --test data/test \
    --features 'MedInc HouseAge AveRooms AveBedrms Population AveOccup Latitude Longitude' \
    --target_variable target

## SageMaker Training

To run your script in a training job, first we need to upload the data somewhere SageMaker can access it: Typically this will be [Amazon S3](https://aws.amazon.com/s3/).

### Creating data input "channels" (copy to S3)

Note that the number and naming of multiple data "channels" for SageMaker is up to you: You don't need to have exactly 2, and they don't need to be called "train" and "test".

The data could be already store in S3 and it doesn't need to be uploaded as part or preparation of the trainin process.

In [ ]:
train_data_s3uri = sess.upload_data(
    path="data/train/train.csv",  # Local source
    bucket=bucket,
    key_prefix=f"{prefix}/train",  # Destination path in S3 bucket
)

test_data_s3uri = sess.upload_data(
    path="data/test/test.csv",  # Local source
    bucket=bucket,
    key_prefix=f"{prefix}/test",  # Destination path in S3 bucket
)

print("Train set URI:", train_data_s3uri)
print("Test set URI:", test_data_s3uri)

### Launching a training job with the Python SDK

With the data uploaded and script prepared, you're ready to configure your SageMaker training job:

In [ ]:
# We use the Estimator from the SageMaker Python SDK
from sagemaker.sklearn.estimator import SKLearn

sklearn_estimator = SKLearn(
    entry_point="main.py",
    source_dir="src",  # To upload the whole folder - alternatively set entry_point="src/main.py"
    role=sagemaker.get_execution_role(),  # Use same IAM role as notebook is currently using
    instance_count=1,
    instance_type="ml.m5.large",
    framework_version="1.0-1",
    base_job_name="rf-scikit",
    metric_definitions=[
        # SageMaker can extract metrics from console logs via Regular Expressions:
        {"Name": "median-AE", "Regex": "AE-at-50th-percentile: ([0-9.]+).*$"},
        {"Name": "RMSE", "Regex": "RMSE: ([0-9.]+).*$"},
        {"Name": "MAE", "Regex": "MAE: ([0-9.]+).*$"},
        {"Name": "R2-score", "Regex": "R2-score: ([0-9.]+).*$"},
    ],
    hyperparameters={
        "n_estimators": 100,
        "min_samples_leaf": 3,
        "features": "MedInc HouseAge AveRooms AveBedrms Population AveOccup Latitude Longitude",
        "target_variable": "target",
        # SageMaker data channels are always folders. Even if you point to a particular object
        # S3URI, you'll need to either: Properly support loading folder inputs in your script; or
        # use extra configuration parameters to identify specific filename(s):
        "train_file": "train.csv",
        "test_file": "test.csv",
    },
    # Optional settings to run with SageMaker Managed Spot:
    max_run=20*60,  # Maximum allowed active runtime (in seconds)
    use_spot_instances=True,  # Use spot instances to reduce cost
    max_wait=30*60,  # Maximum clock time (including spot delays)
)

In [ ]:
sklearn_estimator.fit({"train": train_data_s3uri, "test": test_data_s3uri}, wait=True)

Remember that the training job that we ran is very "light", due to the very small dataset. As such, running locally on the notebook instance results in a faster execution time, compared to SageMaker.

SageMaker takes longer time to run the job because it has to provision the training infrastructure. Since this example training job not very resource-intensive, the infrastructure provisioning process adds more overhead, compared to the training job itself. 

In a real situation, where datasets are large, running on SageMaker can considerably speed up the execution process - and help us optimize costs, by keeping this interactive notebook environment modest and spinning up more powerful training job resources on-demand.

Note that this training job *did not run here on the notebook itself*. You'll be able to see the history in the AWS Console for SageMaker - Training Jobs tab

> ℹ️ **Tip:** There's **no need to re-run** a training job if your notebook kernel restarts or the estimator state is lost for some other reason... You can just *attach* to a previous training job by name - for example:
>
> ```python
> estimator = SKLearn.attach("rf-scikit-2025-01-01-00-00-00-000") #Replace the placeholder with the actual job name
> ```

## Deploy to a real-time endpoint

### Deploy with Python SDK

It's possible to deploy a trained `Estimator` to a SageMaker endpoint for real-time inference in one line of code, with `Estimator.deploy(...)` - which implicitly creates a SageMaker [Model](https://console.aws.amazon.com/sagemaker/home?#/models), [Endpoint Configuration](https://console.aws.amazon.com/sagemaker/home?#/endpointConfig), and [Endpoint](https://console.aws.amazon.com/sagemaker/home?#/endpoints).

For more fine-grained control though, you can choose to create a `Model` object through the SageMaker Python SDK - referencing the `model.tar.gz` produced on Amazon S3 by the training job. This would allow us to, for example:

- Modify environment variables or the Python files used between training and inference
- Import a model trained outside SageMaker that's been packaged to a compatible `model.tar.gz` on Amazon S3

We'll demonstrate the longer route here:

In [ ]:
sklearn_estimator.latest_training_job.wait(logs="None")  # Check the job is finished

# describe() here is equivalent to low-level boto3 SageMaker describe_training_job
job_desc = sklearn_estimator.latest_training_job.describe()
model_s3uri = job_desc["ModelArtifacts"]["S3ModelArtifacts"]

print("Model artifact saved at:", model_s3uri)

In [ ]:
from sagemaker.sklearn.model import SKLearnModel

model = SKLearnModel(
    model_data=model_s3uri,
    framework_version="1.0-1",
    py_version="py3",
    role=sagemaker.get_execution_role(),
    entry_point="src/main.py", # This could be a separate script for inference if desired
)

In [ ]:
predictor = model.deploy(
    instance_type="ml.c5.large",
    initial_instance_count=1,
)

### Realtime inference

The Predictor class from the SageMaker Python SDK provides a Python wrapper around the endpoint which also handles (configurable) de/serialization of the request and response.

Alternatively for clients which cannot use the SageMaker Python SDK (for example non-Python clients, or Python environments where the PyPI [sagemaker](https://pypi.org/project/sagemaker/) package can't be installed for some reason): **The general AWS SDKs can be used to call the lower-level [SageMaker InvokeEndpoint API](https://docs.aws.amazon.com/sagemaker/latest/APIReference/API_runtime_InvokeEndpoint.html).**

In [ ]:
# Por comodidad cargamos el dataset original para filtrar el dataset de test
from sklearn.datasets import fetch_california_housing

data = fetch_california_housing()
testX = pd.read_csv("./data/test/test.csv")

# The SKLearnPredictor does the serialization from pandas for us
print(predictor.predict(testX[data.feature_names]))

### Delete endpoint

While training job infrastructure is started on-demand and terminated as soon as the job stops, endpoints are live until we turn them off. Delete unused endpoints to prevent ongoing costs:

In [ ]:
predictor.delete_endpoint(delete_endpoint_config=True)

## Batch inference

Above we saw how you can deploy your trained model to a real-time API, But what if you want to process a whole batch of data at once? There's no need to manually orchestrate sending this data through an endpoint: You can use [SageMaker Batch Transform](https://docs.aws.amazon.com/sagemaker/latest/dg/batch-transform.html).

Like with training, your input data for a batch transform job needs to be accessible to SageMaker (i.e. uploaded to Amazon S3) and the result will be stored to S3. The compute infrastructure spun up for the job will be released as soon as the data is processed.

Unlike with training, the input data in S3 needs to match the format your model expects for *inference*. This means we'll need to remove `target`, any unused features, and also column headers (although we could have instead overridden `input_fn` in `main.py` (or a separa custom inference script) to make our model handle more input shapes).

In [ ]:
testX[data.feature_names].to_csv("data/transform_input.csv", header=False, index=False)

transform_input_s3uri = sess.upload_data(
    path="data/transform_input.csv",  # Local source
    bucket=bucket,
    key_prefix=prefix,  # Destination path in S3 bucket
)

With the input data uploaded, you're ready to run a transform job using the `model` from before:

In [ ]:
transformer = model.transformer(
    instance_count=1,
    instance_type="ml.m5.xlarge",
    # Input Parameters:
    strategy="MultiRecord",  # Batch multiple records per request to the endpoint
    max_payload=2,  # Max 2MB payload per request
    max_concurrent_transforms=2,  # 2 concurrent request threads per instance
    # Output Parameters:
    output_path=f"s3://{bucket}/{prefix}/sklearn-transforms",
    accept="text/csv",  # Request CSV output format
    assemble_with="Line",  # Records in CSV output are newline-separated
)

In [ ]:
transformer.transform(
    transform_input_s3uri,
    content_type="text/csv",  # Input files are CSV format
    split_type="Line",  # Interpret each line of the CSV as a separate record
    join_source="Input",  # Bring input features through to the output file
    wait=True,  # Keep this notebook blocked until the job completes
    logs=True,  # Stream logs to the notebook
)

For each input object in S3, Batch Transform will generate a similar object under the output folder with `.out` appended to the file name. In our simple example, there was just one input CSV so there will be one `csv.out` result file:

In [ ]:
job_desc = sm_boto3.describe_transform_job(TransformJobName=transformer.latest_transform_job.name)
output_s3uri = job_desc["TransformOutput"]["S3OutputPath"]

# pd.read_csv() can take an "s3://.../.../" folder, but doesn't like that our Batch Transform
# results have .csv.out extension instead of .csv: So instead manually specify which file we want:
!echo "Output folder contents:" && aws s3 ls {output_s3uri}/

input_filename = transform_input_s3uri.rpartition("/")[2]
output_file_s3uri = f"{output_s3uri}/{input_filename}.out"

print(f"\nReading {output_file_s3uri} from S3")
pd.read_csv(output_file_s3uri, names=data.feature_names + ["prediction"])

## Conclusions

In this notebook, we saw an example of:

- Running your own Scikit-Learn-based model training script as a SageMaker training job, with configurable parameters and output accuracy metrics
- Deploying the trained model to a real-time inference API
- Using the model for batch inference

SageMaker took care of the model serving stack for us with no boilerplate code required: Just define override functions if needed (like `input_fn` and `model_fn`) to customize the default behaviour.

At training time, our script read parameters from the command line arguments and environment variables provided through SageMaker - and loaded data from local folder because download from S3 is taken care of by SageMaker too.

By using the SageMaker APIs (instead of just working locally in the notebook), we can improve the traceability and reproducibility of experiments; optimize our compute resource usage; and accelerate the path from trained model to production deployment.